# 01: Estimates check

**Question:** Are any of TriMet's recent monthly values FTA estimates rather than true agency-reported figures?

**Why it matters:** FTA's adjusted ridership file fills in missing recent UPT and VRM values with growth-rate estimates. The raw file does not. A value that appears in the adjusted file but not the raw file is an estimate.

**Inputs** (not committed; see the README for download instructions)
- `data/raw/ntd_monthly_complete_2026-11-15.xlsx` (adjusted)
- `data/raw/ntd_monthly_raw_2026-11-15.xlsx` (raw)

**Method:** Filter both files to TriMet, reshape to one row per mode, type of service, and month, then merge and compare for Jan 2019 onward. I check UPT and VRH, the two measures needed for trips per revenue hour.

**Output:** No files. The findings are recorded at the bottom of this notebook.

## Set up and loader

In [1]:
import re
import pandas as pd

ADJ_PATH = "data/raw/ntd_monthly_complete_2026-11-15.xlsx"
RAW_PATH = "data/raw/ntd_monthly_raw_2026-11-15.xlsx"
AGENCY = "Tri-County Metropolitan Transportation District of Oregon"
WINDOW_START = "2019-01-01"


def load_trimet_long(path, sheet):
    df = pd.read_excel(path, sheet_name=sheet)
    df = df[df["Agency"].astype(str).str.contains(AGENCY, na=False)]
    month_cols = [col for col in df.columns if re.fullmatch(r"\d{1,2}/\d{4}", str(col))]
    long = df.melt(
        id_vars=["Mode", "TOS"],
        value_vars=month_cols,
        var_name="month",
        value_name="value",
    )
    long["month"] = pd.to_datetime(long["month"].astype(str), format="%m/%Y")
    long["value"] = pd.to_numeric(
        long["value"].astype(str).str.replace(",", "").str.strip(), errors="coerce"
    )
    return long

## Comparison function

This merges the adjusted and raw data for one sheet (e.g. UPT) and flags two things:
- Estimated: a value in the adjusted file but not the raw file.
- Changed: a value present in both files but different.

It also prints each row's first and last months with data as a sanity check.

In [2]:
def compare_sheet(sheet):
    adj = load_trimet_long(ADJ_PATH, sheet)
    raw = load_trimet_long(RAW_PATH, sheet)

    m = adj.merge(
        raw, on=["Mode", "TOS", "month"], how="outer", suffixes=("_adj", "_raw")
    )
    m = m[m["month"] >= WINDOW_START]

    estimated = m[m["value_adj"].notna() & m["value_raw"].isna()]
    changed = m[
        m["value_adj"].notna()
        & m["value_raw"].notna()
        & (m["value_adj"] != m["value_raw"])
    ]

    print(f"Sheet: {sheet}")
    print(f"Mode/TOS rows: {adj.groupby(['Mode', 'TOS']).ngroups}")
    print(f"Estimated months: {len(estimated)}")
    print(f"Changed months: {len(changed)}")

    print("\nFull history (adjusted file):")
    print(
        adj.dropna(subset=["value"])
        .groupby(["Mode", "TOS"])["month"]
        .agg(first_month="min", last_month="max", months_with_data="count")
    )

    print(f"\nAnalysis window (from {WINDOW_START[:7]}):")
    print(
        m.dropna(subset=["value_adj"])
        .groupby(["Mode", "TOS"])["month"]
        .agg(first_month="min", last_month="max", months_with_data="count")
    )
    print()

    return adj, raw, m, estimated, changed

## UPT
Unlinked passenger trips

In [3]:
upt_adj, upt_raw, upt_m, upt_est, upt_chg = compare_sheet("UPT")

Sheet: UPT
Mode/TOS rows: 8
Estimated months: 0
Changed months: 0

Full history (adjusted file):
         first_month last_month  months_with_data
Mode TOS                                         
CR   PT   2009-02-01 2011-12-01                35
DR   PT   2002-01-01 2026-09-01               297
     TN   2023-05-01 2026-09-01                41
     TX   2011-01-01 2026-09-01               189
LR   DO   2002-01-01 2026-09-01               297
MB   DO   2002-01-01 2026-09-01               297
SR   DO   2012-01-01 2012-12-01                12
YR   PT   2012-01-01 2026-09-01               177

Analysis window (from 2019-01):
         first_month last_month  months_with_data
Mode TOS                                         
DR   PT   2019-01-01 2026-09-01                93
     TN   2023-05-01 2026-09-01                41
     TX   2019-01-01 2026-09-01                93
LR   DO   2019-01-01 2026-09-01                93
MB   DO   2019-01-01 2026-09-01                93
YR   PT   2019-01-01

**UPT results:** 8 mode/TOS rows. 0 estimated months and 0 changed months. DR/TN has 41 months in the window because it begins in May 2023.

## VRH
Vehicle revenue hours

In [4]:
vrh_adj, vrh_raw, vrh_m, vrh_est, vrh_chg = compare_sheet("VRH")

Sheet: VRH
Mode/TOS rows: 8
Estimated months: 0
Changed months: 0

Full history (adjusted file):
         first_month last_month  months_with_data
Mode TOS                                         
CR   PT   2009-02-01 2011-12-01                35
DR   PT   2002-01-01 2026-09-01               297
     TN   2023-05-01 2026-09-01                41
     TX   2011-01-01 2026-09-01               189
LR   DO   2002-01-01 2026-09-01               297
MB   DO   2002-01-01 2026-09-01               297
SR   DO   2012-01-01 2012-12-01                12
YR   PT   2012-01-01 2026-09-01               177

Analysis window (from 2019-01):
         first_month last_month  months_with_data
Mode TOS                                         
DR   PT   2019-01-01 2026-09-01                93
     TN   2023-05-01 2026-09-01                41
     TX   2019-01-01 2026-09-01                93
LR   DO   2019-01-01 2026-09-01                93
MB   DO   2019-01-01 2026-09-01                93
YR   PT   2019-01-01

## Conclusion

For both UPT and VRH in the analysis window (January 2019 to September 2026) no TriMet values are FTA estimates and none differ between the adjusted and raw files. Both sheets contain the same 8 mode/TOS rows with identical date ranges, so they can be joined directly on mode, type of service, and month.

**Active in the analysis window:** MB (bus), LR (MAX), DR/PT, DR/TX, and DR/TN, and YR. DR/TN starts in May 2023, so it has 41 months instead of 93.

**Inactive, no data in the window:** CR (ended Dec 2011) and SR (Jan to Dec 2012 only).

**Limitations**
- VRM was not checked.
- This reflects the files as downloaded.
- "Agency-reported" means TriMet submitted the numbers, not that they are final or guaranteed accurate.
